# Chi-squared baseline

Runs the consolidated chi2 baseline (`src/models/chi2_baseline.py`) on a test
file, writes the prediction h5 in the format `src/analysis` expects for chi2
baselines, shows the QA (chi2 distributions, purity, ROC), and optionally
renders the group purity/efficiency curves against a SPANet prediction.

Equivalent one-liner from the repo root:
```bash
python -m src.models.chi2_baseline --test-file <test.h5> --out-file <pred.h5> --plot-dir plots/chi2_qa
```

In [ ]:
# --- configuration -------------------------------------------------------
TEST_FILE = "/path/to/tt_hadronic_testing_SLIMMED.h5"   # truth + inputs
PRED_FILE = "tt_hadronic_chi2_baseline.h5"              # output written here
SPANET_PRED = None   # optional: SPANet prediction h5 for the comparison plots
N_TOPS = 2
PLOT_DIR = "plots/chi2_qa"

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root when run from notebooks/

import h5py
from src.models.chi2_baseline import (
    load_jets, load_boosted, resolved_chi2, boosted_chi2,
    write_predictions, qa_report,
)

with h5py.File(TEST_FILE, "r") as f:
    jets = load_jets(f)
    fjets = load_boosted(f)
    print(f"{len(jets):,} events")
    resolved = resolved_chi2(jets, N_TOPS)
    boosted = boosted_chi2(fjets, N_TOPS)
    write_predictions(PRED_FILE, f, resolved, boosted, N_TOPS)
    qa_report(f, resolved, boosted, N_TOPS, PLOT_DIR)

## QA plots

In [ ]:
from IPython.display import IFrame, display
for arm in ("resolved", "boosted"):
    for kind in ("distributions", "roc"):
        display(IFrame(f"{PLOT_DIR}/chi2_{arm}_{kind}.pdf", width=900, height=420))

## Group purity/efficiency curves (chi2 vs SPANet)

Uses the standard analysis exactly as `reports/run_analysis.py` does; the chi2
prediction file is picked up automatically through the analysis' chi2 code
path (no `detection_probability` datasets present).

In [ ]:
from src.analysis.plot import plot_pur_eff_w_dict

plot_dict = {"chi2_45_20": PRED_FILE}  # tag encodes boosted/resolved cuts
if SPANET_PRED:
    plot_dict["SPAtop"] = SPANET_PRED

plot_pur_eff_w_dict(plot_dict, TEST_FILE, save_path=PLOT_DIR, proj_name="SPAtop_chi2")